# 01. Asset Price History - 탐색적 데이터 분석 (EDA)

asset-radar가 수집한 자산 가격 데이터의 품질, 분포, 수집 패턴을 탐색한다.

**대상 자산**: BTC, ETH (Upbit), 삼성전자, SK하이닉스 (KIS), AAPL, NVDA (Alpha Vantage), Gold (GoldAPI)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from db import load_all_prices, get_asset_list

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. 데이터 개요

In [ ]:
# 자산 목록 및 데이터 규모
asset_list = get_asset_list()
print(f"총 자산 수: {len(asset_list)}")
print(f"총 데이터 행 수: {asset_list['row_count'].sum():,}")
print()
asset_list

In [ ]:
# 전체 가격 데이터 로드
df = load_all_prices()
print(f"Shape: {df.shape}")
print(f"\n컬럼별 타입:")
print(df.dtypes)
print(f"\nNull 수:")
print(df.isnull().sum())
df.head()

In [ ]:
# 자산별 데이터 수 막대 차트
counts = df.groupby(['source', 'symbol']).size().reset_index(name='count')
counts['label'] = counts['source'] + ':' + counts['symbol']

fig = px.bar(counts, x='label', y='count', color='source',
             title='자산별 수집된 데이터 포인트 수',
             labels={'label': '자산', 'count': '데이터 수'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 2. 수집 주기 분석

In [ ]:
# 자산별 수집 간격 계산
interval_stats = []

for (source, symbol), group in df.groupby(['source', 'symbol']):
    deltas = group['collected_at'].diff().dropna().dt.total_seconds()
    if len(deltas) == 0:
        continue
    interval_stats.append({
        'source': source,
        'symbol': symbol,
        'median_sec': deltas.median(),
        'mean_sec': deltas.mean(),
        'min_sec': deltas.min(),
        'max_sec': deltas.max(),
        'std_sec': deltas.std(),
        'count': len(deltas)
    })

interval_df = pd.DataFrame(interval_stats)
interval_df['label'] = interval_df['source'] + ':' + interval_df['symbol']
interval_df

In [ ]:
# 수집 간격 히스토그램 (자산별)
fig, axes = plt.subplots(len(df.groupby(['source', 'symbol'])), 1,
                         figsize=(14, 4 * len(interval_df)))
if len(interval_df) == 1:
    axes = [axes]

for ax, ((source, symbol), group) in zip(axes, df.groupby(['source', 'symbol'])):
    deltas = group['collected_at'].diff().dropna().dt.total_seconds()
    ax.hist(deltas, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{source}:{symbol} — 수집 간격 분포')
    ax.set_xlabel('간격 (초)')
    ax.set_ylabel('빈도')
    ax.axvline(deltas.median(), color='red', linestyle='--', label=f'중앙값: {deltas.median():.1f}s')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 갭 탐지: 중앙값의 2배 이상인 간격
print("=== 데이터 수집 갭 탐지 (중앙값 × 2 초과) ===")
print()

for (source, symbol), group in df.groupby(['source', 'symbol']):
    deltas = group['collected_at'].diff()
    median_delta = deltas.median()
    threshold = median_delta * 2
    gaps = deltas[deltas > threshold]

    print(f"{source}:{symbol}")
    print(f"  수집 중앙 간격: {median_delta}")
    print(f"  갭 수 (>{threshold}): {len(gaps)}")
    if len(gaps) > 0:
        print(f"  최대 갭: {gaps.max()}")
        print(f"  갭 비율: {len(gaps)/len(deltas.dropna())*100:.1f}%")
    print()

## 3. 가격 분포

In [ ]:
# 자산별 가격 기술통계
price_stats = df.groupby(['source', 'symbol', 'quote_currency'])['price'].describe()
price_stats

In [ ]:
# 자산별 가격 박스플롯
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# KRW 자산
krw_df = df[df['quote_currency'] == 'KRW']
if not krw_df.empty:
    krw_df.boxplot(column='price', by=['source', 'symbol'], ax=axes[0], rot=45)
    axes[0].set_title('KRW 자산 가격 분포')
    axes[0].set_ylabel('가격 (KRW)')

# USD 자산
usd_df = df[df['quote_currency'] == 'USD']
if not usd_df.empty:
    usd_df.boxplot(column='price', by=['source', 'symbol'], ax=axes[1], rot=45)
    axes[1].set_title('USD 자산 가격 분포')
    axes[1].set_ylabel('가격 (USD)')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. 데이터 가용성 히트맵

In [ ]:
# 시간 × 날짜 히트맵: 자산별 데이터 포인트 밀도
for (source, symbol), group in df.groupby(['source', 'symbol']):
    group = group.copy()
    group['date'] = group['collected_at'].dt.date
    group['hour'] = group['collected_at'].dt.hour

    pivot = group.groupby(['date', 'hour']).size().unstack(fill_value=0)

    plt.figure(figsize=(16, max(4, len(pivot) * 0.3)))
    sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.5,
                xticklabels=True, yticklabels=True)
    plt.title(f'{source}:{symbol} — 시간대별 데이터 수집 밀도')
    plt.xlabel('시간 (UTC)')
    plt.ylabel('날짜')
    plt.tight_layout()
    plt.show()

## 5. 가격 시계열

In [ ]:
# 전체 자산 가격 시계열 (통화별 이중 Y축)
fig = make_subplots(specs=[[{"secondary_y": True}]])

colors = px.colors.qualitative.Set2
for i, ((source, symbol), group) in enumerate(df.groupby(['source', 'symbol'])):
    is_usd = group['quote_currency'].iloc[0] == 'USD'
    fig.add_trace(
        go.Scatter(x=group['collected_at'], y=group['price'],
                   name=f'{source}:{symbol}',
                   line=dict(color=colors[i % len(colors)])),
        secondary_y=is_usd
    )

fig.update_yaxes(title_text='가격 (KRW)', secondary_y=False)
fig.update_yaxes(title_text='가격 (USD)', secondary_y=True)
fig.update_layout(title='전체 자산 가격 추이', xaxis_title='시간',
                  height=500, hovermode='x unified')
fig.show()

## 6. 데이터 품질 요약

In [ ]:
# 종합 데이터 품질 테이블
quality = []

for (source, symbol), group in df.groupby(['source', 'symbol']):
    deltas = group['collected_at'].diff().dropna().dt.total_seconds()
    median_delta = deltas.median()
    gaps = (deltas > median_delta * 2).sum()

    quality.append({
        '자산': f'{source}:{symbol}',
        '통화': group['quote_currency'].iloc[0],
        '데이터 수': len(group),
        'Null%': f"{group['price'].isnull().mean()*100:.1f}%",
        '수집 시작': group['collected_at'].min().strftime('%Y-%m-%d %H:%M'),
        '수집 종료': group['collected_at'].max().strftime('%Y-%m-%d %H:%M'),
        '중앙 간격': f"{median_delta:.0f}s",
        '갭 수': gaps,
        '갭 비율': f"{gaps/max(len(deltas),1)*100:.1f}%",
        '최소가': f"{group['price'].min():,.2f}",
        '최대가': f"{group['price'].max():,.2f}",
        '평균가': f"{group['price'].mean():,.2f}",
        '표준편차': f"{group['price'].std():,.2f}"
    })

quality_df = pd.DataFrame(quality)
quality_df.style.set_caption('데이터 품질 종합 요약')